# Sprint 2 pipeline (chunk → embed → ChromaDB)

Runs one cleaned document through chunking, OpenAI embedding and storage in the
persisted ChromaDB collection.

**What this notebook is for.** Story 3's Definition of Done says:

> Running ingestion twice in a row is demonstrated live to the tester with the
> collection count shown before and after

Step 1 prints the collection count before and after every run. Run that cell
**twice** and the count must stay the same the second time — that is the whole
demonstration.

Example document: `data/processed/pcbus_working_together`.

## Before you run this

1. You need a `.env` at the repository root containing your OpenAI key:

   ```
   OPEN_AI_API_KEY=sk-...
   ```

   `OPENAI_API_KEY` also works — the code accepts either name.

2. Step 1 makes **live, paid** calls to the OpenAI embeddings API. This document is
   ~104 chunks, which is a single batched request and costs well under a cent.

3. Step 1 **writes to the real vector store** at `vectorstore/`. That is the point —
   the tester needs to see the persisted collection change.

## Config — choose the document

In [ ]:
import sys
from pathlib import Path

# Resolve the repo root so `src` imports work no matter where Jupyter started.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# --- the document to run ---
# DOC_DIR holds the cleaned output produced by the Sprint 1 pipeline.
DOC_DIR = ROOT / "data" / "processed" / "pcbus_working_together"

# Input: the cleaned elements JSON. Read only — never modified by this notebook.
CLEANED_JSON = next(DOC_DIR.glob("*-CLEANED.json"))

# Output: the chunk records. A new derived file alongside the cleaned output,
# the same way *-CLEANED-preview.txt sits next to *-CLEANED.json.
CHUNKS_JSON = CLEANED_JSON.with_name(CLEANED_JSON.name.replace("-CLEANED", "-CHUNKS"))

print("repo root   :", ROOT)
print("cleaned in  :", CLEANED_JSON.name)
print("chunks out  :", CHUNKS_JSON.name)

## Step 1 — Chunk, embed and store

Live, paid OpenAI call. Writes to the persisted collection.

**Run this cell twice.** The first run stores the document; the second must leave the
collection count unchanged. If the count grows on the second run, ingestion is not
idempotent and story 3 fails.

`stale removed` counts records from an earlier run that this run no longer produces —
for example after changing chunk size. It should be `0` on an unchanged re-run.

In [ ]:
from src.embeddings import chunking_script_v2 as pipeline
from src.vectorstore_client import get_collection

# Cleaned elements -> chunk records on disk.
chunks = pipeline.chunk_document(CLEANED_JSON, CHUNKS_JSON)
SOURCE_FILE = chunks[0]["source_file"]

# Chunk records -> embeddings -> ChromaDB.
summary = pipeline.ingest_to_chromadb(CHUNKS_JSON)

# How many records this one document holds, separate from the collection total.
collection = get_collection()
doc_records = len(collection.get(where={"source_file": SOURCE_FILE}, include=[])["ids"])

print("\n" + "=" * 62)
print(f"  collection            : {summary['collection']}")
print(f"  document              : {SOURCE_FILE}")
print(f"  chunks embedded       : {summary['chunks_embedded']}")
print(f"  stale records removed : {summary['stale_removed']}")
print("-" * 62)
print(f"  COUNT BEFORE          : {summary['count_before']}")
print(f"  COUNT AFTER           : {summary['count_after']}")
print("-" * 62)
print(f"  records for this doc  : {doc_records}")
print(f"  pipeline version      : {summary['pipeline_version']}")
print(f"  embedding model       : {summary['embedding_model']}")
print("=" * 62)

if summary["count_before"] == 0:
    print("\nFirst run. Run this cell again — the count must not change.")
elif summary["count_before"] == summary["count_after"]:
    print("\nIDEMPOTENT: the count did not change on this re-run.")
else:
    print("\nCOUNT CHANGED — investigate before signing this story off.")

## Step 2 — Preview the first 10 chunks

The `id` is the deterministic ChromaDB ID: document stem, page, position. It is
computed from the data, so re-running produces exactly the same IDs — which is why
re-ingesting overwrites instead of duplicating.

In [ ]:
import json
import textwrap

chunk_records = json.loads(CHUNKS_JSON.read_text(encoding="utf-8"))
chunk_ids = pipeline.build_chunk_ids(chunk_records)

print(f"total chunks: {len(chunk_records)}\n")

for record, chunk_id in list(zip(chunk_records, chunk_ids))[:10]:
    print("-" * 78)
    print(f"id      : {chunk_id}")
    pages = f"{record['page_start']}–{record['page_end']}" if record.get('page_start') != record.get('page_end') else str(record['page_number'])
    print(f"page    : {pages}    type: {record.get('chunk_type', '-')}")
    print(f"section : {record['section_heading']}")
    print(f"chars   : {len(record['text'])}")
    body = record["text"][:400] + ("..." if len(record["text"]) > 400 else "")
    print(textwrap.fill(body, width=78, initial_indent="    ", subsequent_indent="    "))

## Step 3 — Read the whole cleaned document

Renders the Sprint 1 `*-CLEANED-preview.txt` as Markdown so you can read the source
document straight through and check a chunk's `section_heading` and `page_number`
against it.

In [ ]:
# preview clean version of the document
from pathlib import Path

from IPython.display import Markdown, display

preview_txt = next(Path(DOC_DIR).glob("*-CLEANED-preview.txt"))

blocks = []
for block in preview_txt.read_text().split("\n\n"):
    block = block.strip()
    if not block:
        continue
    _tag, _, body = block.partition("\n")
    blocks.append(body or _tag)

display(Markdown("\n\n".join(blocks)))